# v3.0.0 PD PPG 1D-CNN — Lightweight PPG Model (Supplementary)

A 1D CNN operating on native 40-dim PPGs without resizing to 128.
Tests whether a simpler architecture can exploit PPGs without AST overhead.

In [ ]:
#1 - Imports & data loading
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('/data0/b2ai-voice/3.0.0')
PPG_PATH = ROOT / 'features' / 'ppgs.parquet'
PD_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')

SELECTED_TASKS = [
    'prolonged-vowel', 'glides-high-to-low', 'glides-low-to-high',
    'diadochokinesis-pataka', 'rainbow-passage', 'picture-description',
    'story-recall', 'maximum-phonation-time-1',
]
MIN_TIME_FRAMES = 100

# Load PPG data
pf = pq.ParquetFile(PPG_PATH)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','ppgs','n_frames']).to_pandas())
ppg_df = pd.concat(parts, ignore_index=True)
ppg_df['participant_id'] = ppg_df['participant_id'].astype(str).str.zfill(6)

# Build PD labels
pd_pheno = pd.read_csv(PD_PHEN, sep='\t')
ctrl_pheno = pd.read_csv(CTRL_PHEN, sep='\t')
pd_ids = set(pd_pheno['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_pheno['participant_id'].astype(str).str.zfill(6)) - pd_ids

ppg_df['label'] = np.nan
ppg_df.loc[ppg_df['participant_id'].isin(pd_ids), 'label'] = 1
ppg_df.loc[ppg_df['participant_id'].isin(ctrl_ids), 'label'] = 0
data = ppg_df.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
data = data[(data['task_name'].isin(SELECTED_TASKS)) & (data['n_frames'] >= MIN_TIME_FRAMES)].copy()

print(f'PD: {(data["label"]==1).sum()} recs, Ctrl: {(data["label"]==0).sum()} recs')
print(f'Participants: {data["participant_id"].nunique()}')

In [ ]:
#2 - Process PPGs: (40, T) -> pad/crop to 1024 (no resize)
TARGET_SEQ_LEN = 1024

def process_ppg(ppg_raw, target_len=1024):
    spec = np.stack(ppg_raw).astype(np.float32)
    n_phonemes, time_len = spec.shape
    if time_len < target_len:
        spec = np.pad(spec, ((0, 0), (0, target_len - time_len)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

X_list = []
for idx, row in tqdm(data.iterrows(), total=len(data), desc='Processing PPGs'):
    X_list.append(process_ppg(row['ppgs'], TARGET_SEQ_LEN))

X_raw = np.stack(X_list)  # (N, 40, 1024)
y_raw = data['label'].values
participants_raw = data['participant_id'].values
print(f'Shape: {X_raw.shape}')

In [ ]:
#3 - 1D CNN Model
class PPG1DCNN(nn.Module):
    def __init__(self, num_classes=2, in_channels=40):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 128, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.conv3 = nn.Conv1d(256, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (B, 40, 1024)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool1d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool1d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

class PPGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model_test = PPG1DCNN().to(device)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'1D CNN parameters: {total_params:,}')
del model_test

In [ ]:
#4 - 5-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
from scipy import stats
import copy, time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])
print(f'Participants: {len(unique_participants)}, PD: {int(participant_labels.sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()

def evaluate_fold_1d(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(batch['inputs'].to(device))
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs); all_labels.extend(batch['labels'].numpy()); all_parts.extend(batch['participant'])
    all_probs, all_labels, all_parts = np.array(all_probs), np.array(all_labels), np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs = np.array([all_probs[all_parts == p].mean() for p in unique_parts])
    part_labels = np.array([all_labels[all_parts == p][0] for p in unique_parts])
    return part_probs, part_labels, unique_parts

total_start = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    train_parts = unique_participants[train_idx]; val_parts = unique_participants[val_idx]
    train_mask = np.isin(participants_raw, train_parts); val_mask = np.isin(participants_raw, val_parts)

    X_tr = X_raw[train_mask]; y_tr = y_raw[train_mask]; p_tr = participants_raw[train_mask]
    X_va = X_raw[val_mask]; y_va = y_raw[val_mask]; p_va = participants_raw[val_mask]

    # Normalize
    fold_mean, fold_std = X_tr.mean(), X_tr.std()
    X_tr_n = (X_tr - fold_mean) / (fold_std + 1e-8)
    X_va_n = (X_va - fold_mean) / (fold_std + 1e-8)

    train_ds = PPGDataset(X_tr_n, y_tr, p_tr, augment=True)
    val_ds = PPGDataset(X_va_n, y_va, p_va, augment=False)
    cc = np.bincount(y_tr); sw = (1.0 / cc)[y_tr]
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, sampler=torch.utils.data.WeightedRandomSampler(sw, len(sw)), num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

    model = PPG1DCNN(num_classes=2, in_channels=40).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))

    best_score, best_state, patience_counter = 0, None, 0
    for epoch in range(30):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch['inputs'].to(device)), batch['labels'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        scheduler.step()
        pp, pl, _ = evaluate_fold_1d(model, val_loader, device)
        if len(np.unique(pl)) > 1:
            auc = roc_auc_score(pl, pp)
            fpr, tpr, thr = roc_curve(pl, pp)
            f1_opt = f1_score(pl, (pp >= thr[np.argmax(tpr-fpr)]).astype(int), zero_division=0)
        else: auc, f1_opt = 0.5, 0.0
        score = 0.4*auc + 0.6*f1_opt
        if score > best_score + 0.01:
            best_score = score; best_state = copy.deepcopy(model.state_dict()); patience_counter = 0
        else: patience_counter += 1
        if patience_counter >= 10: break

    model.load_state_dict(best_state)
    pp, pl, vpids = evaluate_fold_1d(model, val_loader, device)
    for i, pid in enumerate(vpids):
        all_oof_probs[np.where(unique_participants == pid)[0][0]] = pp[i]
    fold_auc = roc_auc_score(pl, pp)
    fpr, tpr, thr = roc_curve(pl, pp); ot = thr[np.argmax(tpr-fpr)]
    po = (pp >= ot).astype(int)
    fold_results.append({'fold': fold+1, 'auc': float(fold_auc),
        'f1_opt': float(f1_score(pl, po, zero_division=0)),
        'recall_opt': float(recall_score(pl, po, zero_division=0)),
        'precision_opt': float(precision_score(pl, po, zero_division=0))})
    print(f'Fold {fold+1}: AUC={fold_auc:.4f}')
    del model; torch.cuda.empty_cache()

print(f'\nTotal time: {(time.time()-total_start)/60:.1f} min')

In [ ]:
#5 - Summary
aucs = [r['auc'] for r in fold_results]
oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thr = roc_curve(all_oof_labels, all_oof_probs)
oof_thresh = thr[np.argmax(tpr-fpr)]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)

print(f'Mean fold AUC: {np.mean(aucs):.4f} +/- {np.std(aucs, ddof=1):.4f}')
print(f'OOF AUC: {oof_auc:.4f}')
print(f'OOF F1:  {f1_score(all_oof_labels, oof_preds, zero_division=0):.4f}')

np.savez(str(RESULTS_DIR / 'ppg_1dcnn_pd_v3_cv_results.npz'),
    oof_probs=all_oof_probs, oof_labels=all_oof_labels, participant_ids=unique_participants,
    fold_aucs=np.array(aucs), oof_auc=np.array(oof_auc))
print(f'Saved: {RESULTS_DIR}/ppg_1dcnn_pd_v3_cv_results.npz')